# Week 4: LangChain Fundamentals

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week04-langchain.html

**How to use this notebook in Google Colab**

1. Sign in with a Google account. Choose **File → Save a copy in Drive** if you want to keep your changes.
2. This lesson runs a small language model inside the notebook: choose **Runtime → Change runtime type → T4 GPU** before you run anything. Without a GPU the code still works, but each answer takes much longer.
3. Run the cells from the top with **Shift+Enter**, or **Runtime → Run all**. Later examples reuse variables from earlier ones, so do not skip cells. The first time, Colab asks whether to run a notebook that comes from GitHub: choose **Run anyway**.
4. The first two parts of this lesson need no API key. Only the optional hosted-model cell and the last part, *Chat with Databases*, call OpenAI with your own key: click the 🔑 **Secrets** icon in the left bar, add `OPENAI_API_KEY` and switch on *Notebook access*. Never type a key into a cell. With **Run all**, the cells of that last part stop with an error if no key was added; everything before them has already run.

In [ ]:
# The library versions this lesson was tested with (takes a minute or two)
%pip install -q "langchain==1.5.2" "langchain-core==1.8.4" "langchain-community==0.4.2" "langchain-text-splitters==1.1.3" "langchain-huggingface==1.2.3" "langchain-openai==1.8.5" "langgraph==1.2.24" "sentence-transformers" "pypdf" "beautifulsoup4"

In [ ]:
# Reads your API keys from the Colab Secrets panel (🔑 in the left bar), if you added any
import os
try:
    from google.colab import userdata
    for _key in ("OPENAI_API_KEY", "TAVILY_API_KEY", "ANTHROPIC_API_KEY", "HF_TOKEN"):
        try: os.environ[_key] = userdata.get(_key)
        except Exception: pass          # this key was not added: fine if the lesson does not use it
except ImportError:
    pass                                # not running in Colab: keys come from your .env file

In [ ]:
# Imports that some of the lesson's code fragments assume
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

## 2. A Chat Model Without an API Key

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
if torch.cuda.is_available():
    model = model.half().to("cuda")          # half precision on the GPU: less memory, faster

generator = pipeline("text-generation", model=model, tokenizer=tokenizer,
                     max_new_tokens=300, do_sample=False, return_full_text=False)
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator))

print("Running on:", "GPU" if torch.cuda.is_available() else "CPU")
print(llm.invoke("In one sentence, what is LangChain?").content)

## Optional: a hosted model instead

In [ ]:
USE_HOSTED_MODEL = False          # change to True only if you have an API key

if USE_HOSTED_MODEL:
    # The key is read from Colab's Secrets panel (the key icon in the left bar), under the name OPENAI_API_KEY.
    # Model names change often: take a current name from the provider's own list.
    from langchain.chat_models import init_chat_model
    llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

## 3. Prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate the user's text into {language}. Reply with the translation only."),
    ("human", "{text}"),
])

messages = translate_prompt.format_messages(
    language="Arabic",
    text="Abdulkarim Albanna is from Jordan and works at the University of Petra.",
)
for m in messages:
    print(f"{m.type}: {m.content}")
print("Input variables:", translate_prompt.input_variables)

In [ ]:
reply = llm.invoke(messages)
print(reply.content)

## The email assistant

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "You work in customer service for a software company. Write a short, polite reply "
               "to the customer's email. Write the reply in {language}."),
    ("human", "{email}"),
])

messages = email_prompt.format_messages(
    language="Arabic",
    email="اسمي عبدالكريم البنا، أود أن أسأل عن خدمات الشركة للمطورين.",
)
print(llm.invoke(messages).content)

## 4. Chains and the Pipe Operator

In [ ]:
from langchain_core.output_parsers import StrOutputParser

translate_chain = translate_prompt | llm | StrOutputParser()

print(translate_chain.invoke({"language": "French", "text": "Good morning, how are you?"}))

In [ ]:
# batch: several inputs in one call
results = translate_chain.batch([
    {"language": "German", "text": "Where is the library?"},
    {"language": "Spanish", "text": "Where is the library?"},
])
print(results)

# stream: receive the reply piece by piece, as a chat window shows it
for piece in translate_chain.stream({"language": "Arabic", "text": "The lecture starts at ten."}):
    print(piece, end="", flush=True)

## 5. Structured Output

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Review(BaseModel):
    gift: bool = Field(description="Was the item bought as a gift for someone else?")
    delivery_days: int = Field(description="Number of days the delivery took; -1 if the review does not say")
    price_value: list[str] = Field(description="Sentences or phrases about the price or the value of the item")

parser = PydanticOutputParser(pydantic_object=Review)

In [ ]:
print(parser.get_format_instructions())

In [ ]:
review = parser.parse('{"gift": true, "delivery_days": 5, "price_value": ["بسعر ٢٠ دينار"]}')
print(review)
print(review.delivery_days + 1)      # a real integer, not the text "5"

In [ ]:
review_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract information from the customer review.\n{format_instructions}"),
    ("human", "{review}"),
]).partial(format_instructions=parser.get_format_instructions())

review_chain = review_prompt | llm | parser

from langchain_core.exceptions import OutputParserException

try:
    result = review_chain.invoke({"review": "اشتريت هدية لصديقي أحمد بسعر ٢٠ دينار، واستغرق التوصيل أكثر من ٥ أيام."})
    print(result)
    print("Gift?", result.gift, "| Days:", result.delivery_days)
except OutputParserException as error:
    print("The model's reply did not fit the Review class:\n", error)

## 6. Sequential Chains

In [ ]:
html_prompt = ChatPromptTemplate.from_template(
    "You are a web developer. Write a short HTML page for a program that teaches children "
    "programming. The program is: {name}. Reply with the HTML code only."
)
python_prompt = ChatPromptTemplate.from_template(
    "You are a Python developer. Convert the following HTML page into a short Python program "
    "that does the same job, then explain the program in three sentences.\n\n{code}"
)

html_chain = html_prompt | llm | StrOutputParser()
python_chain = python_prompt | llm | StrOutputParser()

overall_chain = {"code": html_chain} | python_chain

print(overall_chain.invoke({"name": "add two numbers"}))

## Several inputs and outputs

In [ ]:
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    RunnablePassthrough.assign(code=html_chain)
    | RunnablePassthrough.assign(explanation=python_chain)
)

result = full_chain.invoke({"name": "add two numbers"})
print(result.keys())
print(result["code"][:300])

## 7. Routing

In [ ]:
teacher = {
    "physics": "You are a physics professor. Explain concisely and name the law or principle involved.",
    "math": "You are a mathematician. Break the problem into steps and solve them in order.",
    "history": "You are a historian. Give dates and causes, and say how certain the evidence is.",
    "computer science": "You are a computer scientist. Explain the idea, then give its time and space cost.",
}
subject_prompts = {
    subject: ChatPromptTemplate.from_messages([("system", instruction), ("human", "{input}")])
    for subject, instruction in teacher.items()
}
default_prompt = ChatPromptTemplate.from_messages([("human", "{input}")])

router_prompt = ChatPromptTemplate.from_template(
    "Which subject does this question belong to? Choose one of: physics, math, history, "
    "computer science, other. Reply with the subject only.\n\nQuestion: {input}"
)
router_chain = router_prompt | llm | StrOutputParser()

print(router_chain.invoke({"input": "What is black body radiation?"}))

In [ ]:
from langchain_core.runnables import RunnableLambda

def pick_prompt(info):
    subject = info["subject"].strip().lower()
    for name, prompt in subject_prompts.items():
        if name in subject:
            return prompt
    return default_prompt                      # no subject matched

routed_chain = (
    RunnablePassthrough.assign(subject=router_chain)
    | RunnableLambda(pick_prompt)
    | llm
    | StrOutputParser()
)

for question in ["What is black body radiation?",
                 "What are greedy algorithms?",
                 "What should I cook for dinner?"]:
    print("Q:", question)
    print("A:", routed_chain.invoke({"input": question}), "\n")

## 4. Step 1: Load

In [ ]:
import pandas as pd
from langchain_core.documents import Document

BASE = "https://raw.githubusercontent.com/megagonlabs/SubjQA/master/SubjQA/electronics/splits/"
subjqa = pd.concat([pd.read_csv(BASE + f"{split}.csv") for split in ("train", "dev", "test")])

tablet = subjqa[subjqa["item_id"] == "B0074BW614"]          # the product used in the textbook
reviews = tablet.drop_duplicates("review_id")

docs = [
    Document(page_content=row.review.replace("ANSWERNOTFOUND", "").strip(),
             metadata={"review_id": row.review_id, "item_id": row.item_id})
    for row in reviews.itertuples()
]

print("Rows in the electronics data:", len(subjqa), "| different reviews:", subjqa["review_id"].nunique())
print("Reviews of the tablet:", len(docs))
print(docs[0].page_content[:300])
print(docs[0].metadata)

## Loading a PDF and a web page

In [ ]:
import os
os.environ.setdefault("USER_AGENT", "606363-course-notebook")   # a web client should name itself
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

# The RAG paper of Lewis et al. (2020), and this course's syllabus page
PDF_SOURCE = "https://arxiv.org/pdf/2005.11401"
WEB_SOURCE = "https://zmomani.github.io/ComputingSystemsDS-AI/lessons/syllabus.html"

pdf_pages = PyPDFLoader(PDF_SOURCE).load()                      # one Document per page
first = pdf_pages[0]
print(len(pdf_pages), "pages |", first.metadata.get("source"), "| page", first.metadata.get("page"))
print(first.page_content[:200])

web_docs = WebBaseLoader(WEB_SOURCE).load()                     # one Document for the whole page
print(len(web_docs[0].page_content), "characters from", web_docs[0].metadata.get("source"))

## 5. Step 2: Split

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(chunk_size=26, chunk_overlap=4)
c_splitter = CharacterTextSplitter(chunk_size=26, chunk_overlap=4)

text1 = "abcdefghijklmnopqrstuvwxyz"                                   # 26 characters
text2 = "abcdefghijklmnopqrstuvwxyzabcdefg"                            # 33 characters
text3 = "a b c d e f g h i j k l m n o p q r s t u v w x y z"          # 26 letters with spaces

print(r_splitter.split_text(text1))
print(r_splitter.split_text(text2))
print(r_splitter.split_text(text3))
print(c_splitter.split_text(text3))

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50, add_start_index=True)
chunks = splitter.split_documents(docs)

print(len(docs), "reviews ->", len(chunks), "chunks")
print(chunks[0].metadata)
print(chunks[0].page_content)

## 6. Step 3: Embed

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_ID, encode_kwargs={"normalize_embeddings": True})

vector = embeddings.embed_query("Is it good for reading?")
print(len(vector), "numbers; the first five:", [round(x, 3) for x in vector[:5]])

## 7. Step 4: Store

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(chunks)             # embeds every chunk and stores it
print(len(ids), "chunks stored")

question = "Is it good for reading?"
for doc, score in vector_store.similarity_search_with_score(question, k=4):
    print(f"{score:.3f} | {doc.metadata['review_id'][:8]} | {doc.page_content[:110]!r}")

## 8. Step 5: Retrieve and Generate

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

def format_docs(found):
    return "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(found, start=1))

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the customer reviews below. If the reviews do not "
               "contain the answer, say that you do not know. Answer in at most three sentences.\n\n"
               "Reviews:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("Is it good for reading?"))

## With and without retrieval

In [ ]:
question = "Is it good for reading?"
print("WITHOUT retrieval:\n", llm.invoke(question).content, "\n")
print("WITH retrieval:\n", rag_chain.invoke(question))

In [ ]:
print(rag_chain.invoke("Who won the football World Cup in 2010?"))

## Returning the sources

In [ ]:
answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["documents"]))
    | rag_prompt | llm | StrOutputParser()
)
rag_with_sources = (
    {"documents": retriever, "question": RunnablePassthrough()}
    | RunnablePassthrough.assign(answer=answer_chain)
)

result = rag_with_sources.invoke("How is the battery life?")
print(result["answer"], "\n")
for d in result["documents"]:
    print("source:", d.metadata["review_id"][:8], "| starts at character", d.metadata["start_index"])

## 9. Measuring the Retriever

In [ ]:
test_questions = tablet[["question", "review_id"]].drop_duplicates()
print(len(test_questions), "question-review pairs")

def recall_at(k):
    hits = 0
    for row in test_questions.itertuples():
        found = vector_store.similarity_search(row.question, k=k)
        hits += any(d.metadata["review_id"] == row.review_id for d in found)
    return hits / len(test_questions)

for k in (1, 3, 5, 10):
    print(f"recall@{k}: {recall_at(k):.2f}")

## Setting Up the Environment

In [ ]:
# Install required packages (run in a terminal):
#   pip install langchain langchain-community langchain-openai langgraph sqlalchemy requests

# Import necessary modules
import os
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langgraph.prebuilt import create_react_agent

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

## Database Connection

In [ ]:
# Connect to an SQLite Database
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

# For an existing SQLite database
db_path = "chinook.db"  # Path to your SQLite database
engine = create_engine(f"sqlite:///{db_path}")
db = SQLDatabase(engine)

# Alternatively, for in-memory testing with example data
def get_in_memory_db():
    import sqlite3
    import requests

    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)

    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )

engine = get_in_memory_db()
db = SQLDatabase(engine)

## Creating the LangChain Agent

In [ ]:
# Initialize Language Model
llm = ChatOpenAI(model="gpt-4o", temperature=0)  # You can also use gpt-4o-mini for lower cost

# Initialize SQL Database Toolkit
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Get the tools from the toolkit
tools = toolkit.get_tools()

# Build the system prompt from LangChain's standard SQL-agent prompt
# (the same text as the "langchain-ai/sql-agent-system-prompt" prompt on LangChain Hub)
system_message = SQL_PREFIX.format(dialect="SQLite", top_k=5)

# Create the agent
agent_executor = create_react_agent(
    llm,
    tools,
    prompt=system_message
)

## Django Integration (views.py)

This code is a file of an application, not a notebook step. The cell saves it as `views.py`; run it from a terminal on your computer (or with `!python views.py` here, if it is a script).

In [ ]:
%%writefile views.py
# views.py for Django integration
from django.http import JsonResponse
from django.views.decorators.csrf import csrf_exempt
import json

@csrf_exempt
def chat_with_db(request):
    if request.method == 'POST':
        try:
            data = json.loads(request.body)
            user_message = data.get('message', '')

            # Process the user message with our agent
            response = agent_executor.invoke(
                {"messages": [("user", user_message)]}
            )

            # Extract the final AI message
            ai_message = response["messages"][-1].content

            return JsonResponse({
                'success': True,
                'message': ai_message
            })

        except Exception as e:
            return JsonResponse({
                'success': False,
                'error': str(e)
            })

    return JsonResponse({
        'success': False,
        'error': 'Only POST requests are supported'
    })

## Run in your browser

**Retrieval for RAG: embed chunks, rank by cosine similarity, build the prompt**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = [
    "Transformers use self-attention to relate all tokens in a sequence.",
    "BERT is an encoder-only model pre-trained with masked language modelling.",
    "GPT models are decoder-only and generate text left to right.",
    "Amman is the capital of Jordan and its largest city.",
    "Petra is a historical city in southern Jordan carved into rock.",
    "A vector store returns the chunks whose embeddings are nearest to the query.",
]
question = "How do GPT and BERT models differ?"
k = 2

# TF-IDF stands in for an embedding model here: it matches shared words, whereas a real
# embedding model (e.g. all-MiniLM-L6-v2) also matches meaning, such as "capital" ~ "largest city".
vec = TfidfVectorizer().fit(chunks + [question])
sims = cosine_similarity(vec.transform([question]), vec.transform(chunks))[0]
top = sims.argsort()[::-1][:k]

print("ranking:")
for i in sims.argsort()[::-1]:
    print(f"  {sims[i]:.3f}  {chunks[i]}")

context = "\n".join(f"- {chunks[i]}" for i in top)
prompt = f"Answer using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
print("\nprompt sent to the LLM:\n" + prompt)

**A tiny "chain": prompt template -> model -> output parser (pure Python)**

In [ ]:
def prompt_template(topic):
    return f"List three key facts about {topic}, one per line."

def fake_llm(prompt):                 # stands in for ChatOpenAI / Ollama
    facts = {"MCP": ["Uses JSON-RPC 2.0", "Servers expose tools, resources and prompts", "Transports: stdio and Streamable HTTP"]}
    topic = prompt.split("about ")[1].split(",")[0]
    return "\n".join(facts.get(topic, ["(no facts)"]))

def output_parser(text):
    return [line.strip() for line in text.splitlines() if line.strip()]

def chain(topic):                     # LCEL equivalent: prompt | llm | parser
    return output_parser(fake_llm(prompt_template(topic)))

print(chain("MCP"))

## Test code for the course environment

**A chain without a model: see what each step produces**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Summarize {text} for a {audience}."),
])
echo_model = RunnableLambda(lambda p: AIMessage(content=f"(the model received {len(p.to_messages())} messages)"))

chain = prompt | echo_model | StrOutputParser()
print(prompt.input_variables)
print(chain.invoke({"text": "the LangChain packages", "audience": "first-year student"}))

**Parse a model's JSON reply into a checked object**

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Source(BaseModel):
    title: str = Field(description="Title of the source")
    year: int = Field(description="Year of publication")
    relevant: bool = Field(description="Is the source relevant to the research topic?")

parser = PydanticOutputParser(pydantic_object=Source)
print(parser.parse('{"title": "Attention Is All You Need", "year": 2017, "relevant": true}'))

**Split your own text and inspect the chunks**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = ("RAG has two stages. First the documents are indexed.\n\n"
        "Then a question is answered: the retriever finds the chunks closest to the question, "
        "and the model writes an answer from them.\n\n"
        "A chunk that is too large is cut off by the embedding model. A chunk that is too small loses its context.")

splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=0)
for chunk in splitter.split_text(text):
    print(len(chunk), repr(chunk))

**A vector store in five lines, with a stand-in embedding model**

In [ ]:
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(DeterministicFakeEmbedding(size=64))
store.add_texts(["Petra is a historical city in southern Jordan.",
                 "Amman is the capital of Jordan.",
                 "BERT is an encoder-only transformer."])
for doc, score in store.similarity_search_with_score("Amman is the capital of Jordan.", k=2):
    print(round(score, 3), "|", doc.page_content)

**Load Chinook into an in-memory SQLite DB and inspect it with SQLDatabase (no LLM needed)**

In [ ]:
# 1. Load Chinook into an in-memory SQLite DB and inspect it with SQLDatabase (no LLM needed)
# pip install langchain-community sqlalchemy requests
import sqlite3, requests
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool
from langchain_community.utilities import SQLDatabase

url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.executescript(requests.get(url).text)
engine = create_engine("sqlite://", creator=lambda: conn, poolclass=StaticPool,
                       connect_args={"check_same_thread": False})
db = SQLDatabase(engine)

print(db.dialect)
print(db.get_usable_table_names())
print(db.get_table_info(["Customer", "Invoice"])[:800])

**Run the Process Flow SQL by hand and confirm the "USA, 523.06" result**

In [ ]:
# 2. Run the Process Flow SQL by hand and confirm the "USA, 523.06" result
sql = """
SELECT c.Country, SUM(i.Total) AS TotalSpent
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.Country
ORDER BY TotalSpent DESC
LIMIT 1;
"""
print(db.run(sql))

**Look at the four tools the SQLDatabaseToolkit exposes**

In [ ]:
# 3. Look at the four tools the SQLDatabaseToolkit exposes
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
for t in toolkit.get_tools():
    print(f"{t.name}: {t.description[:90]}...")

**Ask the SQL agent a question and watch every intermediate step**

In [ ]:
# 4. Ask the SQL agent a question and watch every intermediate step
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX
from langgraph.prebuilt import create_react_agent

system_message = SQL_PREFIX.format(dialect="SQLite", top_k=5)
agent_executor = create_react_agent(llm, toolkit.get_tools(), prompt=system_message)

for step in agent_executor.stream(
    {"messages": [("user", "Who are the top 3 best selling artists?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()